# Model Comparison

This notebook mirrors the canonical script pipeline. All models use the shared feature list, preprocessing, and split utilities.

In [ ]:
import numpy as np
import pandas as pd

from cancer_ml_utils import build_model_registry, load_model_dataframe, regression_metrics, split_dataset


In [ ]:
split_strategy = "random"  # change to "drug" or "cell_line" for stricter grouped evaluation
model_df = load_model_dataframe(include_group_columns=split_strategy != "random")
data_split = split_dataset(model_df, split_strategy=split_strategy)

print("Split strategy:", split_strategy)
print("Train:", data_split.X_train.shape)
print("Validation:", data_split.X_val.shape)
print("Test:", data_split.X_test.shape)


In [ ]:
results = []
trained_models = {}

for model_name, model in build_model_registry().items():
    model.fit(data_split.X_train, data_split.y_train)
    val_pred = model.predict(data_split.X_val)
    metrics = regression_metrics(data_split.y_val, val_pred)
    row = {"Model": model_name, **metrics}
    if model_name == "Lasso":
        row["Best Alpha"] = model.named_steps["model"].alpha_
        row["Nonzero Coefficients"] = int(np.count_nonzero(model.named_steps["model"].coef_))
    results.append(row)
    trained_models[model_name] = model

results_df = pd.DataFrame(results).sort_values(
    by=["RMSE", "MAE", "R2"],
    ascending=[True, True, False],
).reset_index(drop=True)

results_df


In [ ]:
best_model_name = results_df.loc[0, "Model"]
best_model = trained_models[best_model_name]
y_test_pred = best_model.predict(data_split.X_test)
test_metrics = regression_metrics(data_split.y_test, y_test_pred)

print("Selected best model:", best_model_name)
print("Final Test Performance")
print("RMSE:", round(test_metrics["RMSE"], 4))
print("MAE :", round(test_metrics["MAE"], 4))
print("R2  :", round(test_metrics["R2"], 4))
